# Function declarations

Every function, class and constant the project uses is defined here, and nothing is
executed. `data_processing.ipynb` and `main.ipynb` each load this notebook with
`%run ./function_declaration.ipynb` and then do all the running.

The sections follow the order of the pipeline:

1. **Data loading**: read the raw NOAA/GLOS files and merge them into one monthly table
2. **Models**: persistence, seasonal climatology, water balance, gradient boosting, hybrid
3. **Evaluation**: rolling-origin backtest, RMSE, bootstrap comparison
4. **Figures**: the five report figures

## 0. Imports and paths

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor

In [ ]:
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").is_dir())
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
FIGURES = ROOT / "figures"

## 1. Data loading

Read the raw NOAA/GLOS files and merge them into one monthly table.

Raw files in `data/raw/` are never modified. Everything here is a pure
read → reshape → merge.

In [ ]:
AREA_MIC_KM2 = 57_800.0
AREA_HUR_KM2 = 59_600.0
AREA_COMBINED_M2 = (AREA_MIC_KM2 + AREA_HUR_KM2) * 1e6

START = "1950-01"

**Precipitation and evaporation files.** One row per year with twelve monthly
columns. We want one long series with one value per month. Values are in mm.

In [ ]:
def read_depth_table(name):
    wide = pd.read_csv(RAW / name, comment="#", skipinitialspace=True)
    wide.columns = range(1, 13)
    long = wide.stack()
    long.index = pd.PeriodIndex(
        [f"{year}-{month:02d}" for year, month in long.index], freq="M"
    )
    return long

**Runoff files.** Already one row per month; keep the flow and the share of the basin that was gauged.

In [ ]:
def read_runoff(name):
    df = pd.read_csv(RAW / name, sep=r"\s*,\s*", engine="python")
    df.index = pd.PeriodIndex(df["YYYY-MM"], freq="M")
    return df[["flow_cms", "fraction_gauged"]]

**Lake levels.** Monthly lake-wide mean level from GLOS. Levels are in metres (IGLD 1985 datum); the second row of the file holds units, not data.

In [ ]:
def read_levels():
    df = pd.read_csv(RAW / "levels_michigan_huron.csv", skiprows=[1])
    idx = pd.PeriodIndex(
        [f"{y}-{m:02d}" for y, m in zip(df["Year"], df["Month"])], freq="M"
    )
    return pd.Series(df["Water_Level_m"].to_numpy(), index=idx)

**Runoff as a depth.** Runoff is a flow (m³/s). Multiply by the seconds in the month
to get a volume, then spread it over the lake surface to get millimetres, the same
unit as precipitation and evaporation.

In [ ]:
def runoff_to_depth_mm(flow_cms):
    seconds_in_month = flow_cms.index.days_in_month * 86_400
    return flow_cms * seconds_in_month / AREA_COMBINED_M2 * 1000.0

**Merge everything.** Michigan and Huron are treated as one lake. Precipitation and
evaporation are depths, so they are averaged by lake area; runoff is a volume, so the
two basins are added. The target `dlevel_mm` is the month-to-month change in level.

In [ ]:
def build_monthly():
    w_mic = AREA_MIC_KM2 / (AREA_MIC_KM2 + AREA_HUR_KM2)
    w_hur = 1.0 - w_mic

    # Precip and evap are depths over each lake -> area-weighted average.
    precip = w_mic * read_depth_table("prc_mic_lake.csv") + w_hur * read_depth_table(
        "prc_hur_lake.csv"
    )
    evap = w_mic * read_depth_table("evaporation_mic.csv") + w_hur * read_depth_table(
        "evaporation_hur.csv"
    )

    # Runoff is a volume flux -> add the two basins, then convert to depth.
    ro_mic = read_runoff("runoff_mic_arm.csv")
    ro_hur = read_runoff("runoff_hur_arm.csv")
    runoff = runoff_to_depth_mm(ro_mic["flow_cms"] + ro_hur["flow_cms"])

    df = pd.DataFrame(
        {
            "level_m": read_levels(),
            "precip_mm": precip,
            "evap_mm": evap,
            "runoff_mm": runoff,
            "frac_gauged_mic": ro_mic["fraction_gauged"],
            "frac_gauged_hur": ro_hur["fraction_gauged"],
        }
    )
    df["nbs_mm"] = df["precip_mm"] - df["evap_mm"] + df["runoff_mm"]
    # Target: change in the monthly mean level from the previous month, in mm.
    df["dlevel_mm"] = df["level_m"].diff() * 1000.0

    df = df.loc[START:]
    # Keep only the span where every series is observed.
    complete = df[["level_m", "precip_mm", "evap_mm", "runoff_mm"]].notna().all(axis=1)
    df = df.loc[: complete[complete].index.max()]
    assert df.notna().all().all(), "unexpected gap inside the analysis period"
    df.index.name = "month"
    return df

## 2. Models

Every model answers the same question: standing at the end of month *t* (the
"origin"), knowing levels and P/E/R up to and including month *t*, what is the level
change `L[t+h] - L[t]` for *h* = 1..6 months ahead?

All models share one interface:

```python
model.fit(train)                     # train: monthly DataFrame, rows before the test year
model.predict(df, origins, horizons) # -> array (n_origins, n_horizons), cumulative change in mm
```

`df` passed to `predict` may contain future rows; a model must only use rows up to
each origin (the perfect-forcing model is the one deliberate exception).

In [ ]:
COMPONENTS = ["precip_mm", "evap_mm", "runoff_mm"]
LEVEL_REF_M = 176.5  # only centres the level term; has no effect on forecasts

**Month of the year.** A small helper: give it a monthly index and it returns the calendar
month number (1–12) of every row. It is used wherever something is averaged by calendar month.

In [ ]:
def month_of(index):
    return index.month.to_numpy()

**Persistence.** The simplest possible forecast: the level stays where it is.

What each part does:

- `fit(train)`: nothing to learn, so it just returns the model. It has to exist because
  the backtest calls `fit` on every model.
- `predict(df, origins, horizons)`: returns zeros, one row per origin and one column per
  horizon, meaning "no change".

In [ ]:
class Persistence:
    def fit(self, train):
        return self

    def predict(self, df, origins, horizons):
        return np.zeros((len(origins), len(horizons)))

**Seasonal climatology.** The lake rises in spring and falls in autumn; forecast the average change for each calendar month.

What each part does:

- `fit(train)`: averages the month-to-month level change for each calendar month over
  the training years, giving twelve numbers.
- `predict(df, origins, horizons)`: for each origin, adds up those averages for the
  coming months. The forecast at horizon *h* is the sum of the first *h* of them.

In [ ]:
class SeasonalClimatology:
    def fit(self, train):
        self.mean_change_ = train["dlevel_mm"].groupby(month_of(train.index)).mean()
        return self

    def predict(self, df, origins, horizons):
        out = np.zeros((len(origins), len(horizons)))
        for i, origin in enumerate(origins):
            steps = [self.mean_change_[(origin + k).month] for k in range(1, max(horizons) + 1)]
            cum = np.cumsum(steps)
            out[i] = [cum[h - 1] for h in horizons]
        return out

**Water balance.** A regression shaped like the water budget: level change = supply
(precipitation, evaporation, runoff) + a monthly constant + a term that lets outflow
grow when the lake is high. It is stepped forward one month at a time.

```
dL[t] = a[month] + sum_c (b0_c * X_c[t] + b1_c * X_c[t-1]) + g * (L[t-1] - ref)
```

for *c* in precipitation, evaporation, runoff.

- Two lags per component because levels are monthly *means*: the change between two
  monthly means straddles two months of supply.
- `a[month] + g * level` stands in for the connecting-channel flows (St. Marys in,
  St. Clair out), which we have no data for; outflow rises with level.
- To forecast, future P/E/R are unknown, so each is replaced by its calendar-month
  climatology plus a decaying share of the latest anomaly (AR(1)).
- With `perfect_forcing=True` the observed future P/E/R are used instead. That is not a
  real forecast, but an upper bound on what this model could do with a perfect weather
  forecast.
- `coefficients()` returns the fitted coefficients as a labelled table.

What each part does:

- `__init__(perfect_forcing)`: remembers whether to use the observed future P/E/R.
- `_design(...)`: builds the regression table: twelve month indicator columns, this
  month's P/E/R, last month's P/E/R, and the previous level minus the reference level.
- `fit(train)`: solves for the coefficients by least squares. It also learns the forcing
  model: the calendar-month average of P/E/R and how strongly an anomaly carries over to
  the next month (`phi_`).
- `predict(df, origins, horizons)`: from each origin, steps forward one month at a time,
  feeding its own predicted level back in and adding up the changes.

In [ ]:
class WaterBalance:
    def __init__(self, perfect_forcing=False):
        self.perfect_forcing = perfect_forcing

    def _design(self, month, x_now, x_prev, level_prev):
        dummies = np.zeros((len(month), 12))
        dummies[np.arange(len(month)), month - 1] = 1.0
        return np.column_stack([dummies, x_now, x_prev, level_prev - LEVEL_REF_M])

    def fit(self, train):
        x = train[COMPONENTS].to_numpy()
        month = month_of(train.index)
        design = self._design(month[1:], x[1:], x[:-1], train["level_m"].to_numpy()[:-1])
        target = train["dlevel_mm"].to_numpy()[1:]
        self.coef_, *_ = np.linalg.lstsq(design, target, rcond=None)

        # Forcing model: monthly climatology + AR(1) on the anomalies.
        self.clim_ = train[COMPONENTS].groupby(month).mean()
        anom = x - self.clim_.loc[month].to_numpy()
        self.phi_ = np.array(
            [np.corrcoef(anom[1:, j], anom[:-1, j])[0, 1] for j in range(len(COMPONENTS))]
        )
        return self

    def predict(self, df, origins, horizons):
        x_all = df[COMPONENTS]
        out = np.zeros((len(origins), len(horizons)))
        for i, origin in enumerate(origins):
            x_prev = x_all.loc[origin].to_numpy()
            anom = x_prev - self.clim_.loc[origin.month].to_numpy()
            level_start = df.at[origin, "level_m"]
            cum = 0.0
            path = {}
            for k in range(1, max(horizons) + 1):
                target = origin + k
                if self.perfect_forcing:
                    x_now = x_all.loc[target].to_numpy()
                else:
                    x_now = self.clim_.loc[target.month].to_numpy() + self.phi_**k * anom
                row = self._design(
                    np.array([target.month]), x_now[None], x_prev[None],
                    np.array([level_start + cum / 1000.0]),
                )
                cum += float(row[0] @ self.coef_)
                path[k] = cum
                x_prev = x_now
            out[i] = [path[h] for h in horizons]
        return out

    def coefficients(self):
        names = [f"month_{m}" for m in range(1, 13)]
        names += [f"{c}_lag0" for c in COMPONENTS] + [f"{c}_lag1" for c in COMPONENTS]
        names += ["level_m"]
        return pd.Series(self.coef_, index=names)

**Features for the ML model.** Everything the model is allowed to know at the end of
a month: the current level, the last three months of level change and supply,
accumulated supply over 3–12 months, and the time of year. Row *t* uses data up to month *t* only.

In [ ]:
def make_features(df):
    f = pd.DataFrame(index=df.index)
    f["level"] = df["level_m"] - LEVEL_REF_M
    for lag in range(3):
        f[f"dlevel_lag{lag}"] = df["dlevel_mm"].shift(lag)
        for c in COMPONENTS:
            f[f"{c[:-3]}_lag{lag}"] = df[c].shift(lag)
    for window in (3, 6, 12):
        f[f"nbs_sum{window}"] = df["nbs_mm"].rolling(window).sum()
    angle = 2 * np.pi * month_of(df.index) / 12
    f["month_sin"] = np.sin(angle)
    f["month_cos"] = np.cos(angle)
    return f

**Feature groups.** The features above bundled into seven named groups, so Figure 4 can
shuffle one whole group at a time.

In [ ]:
# Feature groups used for the permutation-importance figure.
FEATURE_GROUPS = {
    "Season (month)": ["month_sin", "month_cos"],
    "Current level": ["level"],
    "Recent level change": [f"dlevel_lag{k}" for k in range(3)],
    "Precipitation": [f"precip_lag{k}" for k in range(3)],
    "Evaporation": [f"evap_lag{k}" for k in range(3)],
    "Runoff": [f"runoff_lag{k}" for k in range(3)],
    "Accumulated NBS (3-12 mo)": ["nbs_sum3", "nbs_sum6", "nbs_sum12"],
}

**Training target.** The level change from month *t* to month *t+h*, in mm, lined up on
month *t*. It is empty (NaN) where *t+h* runs past the end of the data.

In [ ]:
def _future_change(df, h):
    return (df["level_m"].shift(-h) - df["level_m"]) * 1000.0

**Gradient boosting.** One tree model per horizon, predicting the level change
directly. With `base=WaterBalance()` it becomes the hybrid: the trees only learn what
the water-balance model got wrong.

Hyperparameters were fixed in advance (small, slow-learning, shallow trees for a few
hundred training rows) and were not tuned on the test years.

What each part does:

- `params`: the fixed tree settings.
- `__init__(base)`: remembers the optional base model (none for plain gradient boosting,
  `WaterBalance()` for the hybrid).
- `fit(train)`: builds the features, fits the base model if there is one, then trains one
  tree model for each horizon 1–6 on the rows where features and target both exist. With
  a base model, the target is what the base model got wrong.
- `predict(df, origins, horizons)`: looks up the features at each origin and makes one
  prediction per horizon. With a base model, its forecast is added back on.

In [ ]:
class GradientBoosting:
    params = dict(n_estimators=300, learning_rate=0.03, max_depth=3,
                  subsample=0.8, min_samples_leaf=5, random_state=0)

    def __init__(self, base=None):
        self.base = base

    def fit(self, train):
        self.models_ = {}
        feats = make_features(train)
        ok = feats.notna().all(axis=1)
        if self.base is not None:
            self.base.fit(train)
        for h in range(1, 7):
            y = _future_change(train, h)
            rows = ok & y.notna()
            origins = train.index[rows]
            target = y[rows].to_numpy()
            if self.base is not None:
                target = target - self.base.predict(train, origins, [h])[:, 0]
            self.models_[h] = GradientBoostingRegressor(**self.params).fit(
                feats.loc[origins], target
            )
        return self

    def predict(self, df, origins, horizons):
        # Features are backward-looking, so computing them on the full table
        # does not leak future information into a given origin's row.
        feats = make_features(df).loc[origins]
        out = np.column_stack([self.models_[h].predict(feats) for h in horizons])
        if self.base is not None:
            out = out + self.base.predict(df, origins, horizons)
        return out

**All models**, by the names used in every table and figure.

In [ ]:
def all_models():
    return {
        "Persistence": Persistence(),
        "Seasonal climatology": SeasonalClimatology(),
        "Water balance": WaterBalance(),
        "Gradient boosting": GradientBoosting(),
        "Hybrid (WB + GB residual)": GradientBoosting(base=WaterBalance()),
        "WB, perfect forcing (upper bound)": WaterBalance(perfect_forcing=True),
    }

## 3. Evaluation

In [ ]:
HORIZONS = [1, 2, 3, 4, 5, 6]
FIRST_TEST_YEAR = 1980  # leaves 30 years (1950-79) for the first training set

**Load the merged table.** Read `data/processed/monthly.csv` (written by
`data_processing.ipynb`) back in, with the months as the index.

In [ ]:
def load_monthly():
    df = pd.read_csv(PROCESSED / "monthly.csv", index_col="month")
    df.index = pd.PeriodIndex(df.index, freq="M")
    return df

**Backtest.** Runs the rolling-origin test described above. The result has one row per
(model, target month, horizon) with the observed level, the predicted level and the
error in cm.

In [ ]:
def backtest(df, first_test_year=FIRST_TEST_YEAR, verbose=True):
    rows = []
    for year in range(first_test_year, df.index.max().year + 1):
        train = df.loc[: f"{year - 1}-12"]
        targets = df.loc[f"{year}-01":f"{year}-12"].index
        models = {name: m.fit(train) for name, m in all_models().items()}
        for h in HORIZONS:
            origins = targets - h
            start = df.loc[origins, "level_m"].to_numpy()
            observed = df.loc[targets, "level_m"].to_numpy()
            for name, model in models.items():
                change_mm = model.predict(df, origins, [h])[:, 0]
                rows.append(pd.DataFrame({
                    "model": name, "target": targets, "origin": origins, "horizon": h,
                    "observed_m": observed, "predicted_m": start + change_mm / 1000.0,
                }))
        if verbose:
            print(f"  tested {year}", end="\r")
    preds = pd.concat(rows, ignore_index=True)
    preds["error_cm"] = (preds["predicted_m"] - preds["observed_m"]) * 100.0
    return preds

**RMSE table.** Summarise the backtest as root-mean-square error in cm, with models as
rows and horizons as columns.

In [ ]:
def rmse_table(preds):
    rmse = preds.groupby(["model", "horizon"], sort=False)["error_cm"].apply(
        lambda e: np.sqrt(np.mean(e**2))
    )
    return rmse.unstack("horizon")

**Is a difference between two models real?** Resample whole test years many times
and see how much the RMSE gap moves. The result is
RMSE(a) − RMSE(b) per horizon with a 95% interval. Whole test years are resampled
together because errors within a year (and across overlapping multi-month forecasts) are
strongly correlated.

In [ ]:
def bootstrap_rmse_diff(preds, model_a, model_b, n_boot=2000, seed=0):
    rng = np.random.default_rng(seed)
    out = {}
    for h in HORIZONS:
        sq = {}
        for name in (model_a, model_b):
            p = preds[(preds["model"] == name) & (preds["horizon"] == h)]
            sq[name] = (p["error_cm"] ** 2).groupby(p["target"].dt.year).agg(["sum", "count"])
        years = sq[model_a].index.to_numpy()
        a, b = sq[model_a].to_numpy(), sq[model_b].to_numpy()

        def diff(idx):
            return (np.sqrt(a[idx, 0].sum() / a[idx, 1].sum())
                    - np.sqrt(b[idx, 0].sum() / b[idx, 1].sum()))

        boots = [diff(rng.integers(0, len(years), len(years))) for _ in range(n_boot)]
        lo, hi = np.percentile(boots, [2.5, 97.5])
        out[h] = {"diff_cm": diff(np.arange(len(years))), "ci_low": lo, "ci_high": hi}
    return pd.DataFrame(out).T.rename_axis("horizon")

## 4. Figures

In [ ]:
INK, MUTED, GRID = "#0b0b0b", "#898781", "#e1e0d9"
BLUE, ORANGE, AQUA, VIOLET = "#2a78d6", "#eb6834", "#1baf7a", "#6250d6"
MODEL_STYLE = {  # colour follows the model in every figure
    "Persistence": dict(color=MUTED, ls="-"),
    "Seasonal climatology": dict(color=VIOLET, ls="-"),
    "Water balance": dict(color=BLUE, ls="-"),
    "Gradient boosting": dict(color=ORANGE, ls="-"),
    "Hybrid (WB + GB residual)": dict(color=AQUA, ls="-"),
    "WB, perfect forcing (upper bound)": dict(color=INK, ls="--"),
}
MONTHS = ["J", "F", "M", "A", "M", "J", "J", "A", "S", "O", "N", "D"]

In [ ]:
plt.rcParams.update({
    "font.size": 9, "axes.titlesize": 10, "axes.titleweight": "bold",
    "axes.titlelocation": "left", "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": "#52514e",
    "xtick.color": "#52514e", "ytick.color": "#52514e",
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "lines.linewidth": 2, "legend.frameon": False, "figure.dpi": 100, "savefig.dpi": 200,
    "savefig.bbox": "tight",
})

**Saving a figure.** Write the figure to `figures/` as a PNG, close it, and print the file name.

In [ ]:
def save(fig, name):
    fig.savefig(FIGURES / name)
    plt.close(fig)
    print("wrote figures/" + name)

**Figure 1.** The level record and the average seasonal cycle of supply.

In [ ]:
def fig1_data(df):
    fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.4), width_ratios=[1.7, 1])
    t = df.index.to_timestamp()
    a.plot(t, df["level_m"], color=BLUE, lw=1.3)
    for label, month, dy in [("Record low\nJan 2013", "2013-01", -0.16),
                             ("Jul 2020 high", "2020-07", 0.1)]:
        p = pd.Period(month, "M")
        a.plot(p.to_timestamp(), df.at[p, "level_m"], "o", color=INK, ms=4)
        a.annotate(label, (p.to_timestamp(), df.at[p, "level_m"] + dy), ha="center",
                   va="center", fontsize=8, color=INK)
    a.set_title("Monthly mean level, Lake Michigan-Huron")
    a.set_ylabel("Level (m, IGLD 1985)")
    a.set_ylim(175.3, 177.7)

    clim = df[COMPONENTS].groupby(month_of(df.index)).mean()
    for col, label, color in [("precip_mm", "Precipitation", BLUE),
                              ("evap_mm", "Evaporation", ORANGE),
                              ("runoff_mm", "Runoff", AQUA)]:
        b.plot(range(1, 13), clim[col], color=color, marker="o", ms=3.5, label=label)
    b.set_xticks(range(1, 13), MONTHS)
    b.set_ylim(0, None)
    b.set_title("Average seasonal cycle of supply")
    b.set_ylabel("mm per month over the lake")
    b.legend(loc="upper center", ncol=3, fontsize=8, handlelength=1.2, columnspacing=1,
             bbox_to_anchor=(0.5, -0.12))
    save(fig, "fig1_data_overview.png")

**Figure 2.** Does supply explain level change? Observed level change against the two-month mean of net basin supply.

In [ ]:
def fig2_water_balance(df):
    nbs2 = (df["nbs_mm"] + df["nbs_mm"].shift()) / 2
    ok = nbs2.notna()
    x, y = nbs2[ok].to_numpy(), df.loc[ok, "dlevel_mm"].to_numpy()
    slope, intercept = np.polyfit(x, y, 1)
    r = np.corrcoef(x, y)[0, 1]
    fig, a = plt.subplots(figsize=(5, 4))
    a.scatter(x, y, s=9, color=BLUE, alpha=0.45, lw=0)
    xs = np.array([x.min(), x.max()])
    a.plot(xs, intercept + slope * xs, color=INK, lw=1.5)
    a.axhline(0, color="#c3c2b7", lw=0.8)
    a.set_title("Level change follows net basin supply")
    a.set_xlabel("Net basin supply, mean of this and previous month (mm)")
    a.set_ylabel("Change in monthly mean level (mm)")
    a.text(0.03, 0.97, f"r = {r:.2f}\nslope = {slope:.2f}\nintercept = {intercept:.0f} mm",
           transform=a.transAxes, va="top", fontsize=8.5, color=INK)
    save(fig, "fig2_water_balance_check.png")
    return dict(r=r, slope=slope, intercept=intercept)

**Figure 3.** Forecast error by horizon for every model.

In [ ]:
def fig3_rmse(preds):
    table = rmse_table(preds)
    fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.8))
    for name, style in MODEL_STYLE.items():
        a.plot(HORIZONS, table.loc[name], marker="o", ms=4, label=name, **style)
    a.set_title("Forecast error by horizon")
    a.set_xlabel("Forecast horizon (months ahead)")
    a.set_ylabel("RMSE of level (cm)")
    a.set_ylim(0, None)
    a.legend(fontsize=8, loc="upper left")

    ref = table.loc["Seasonal climatology"]
    for name in ["Water balance", "Gradient boosting", "Hybrid (WB + GB residual)"]:
        b.plot(HORIZONS, 100 * (1 - table.loc[name] / ref), marker="o", ms=4,
               label=name, **MODEL_STYLE[name])
    b.axhline(0, color=VIOLET, lw=1.5)
    b.text(6, 0.6, "seasonal climatology", color="#52514e", fontsize=8, ha="right", va="bottom")
    b.set_title("Error reduction vs. seasonal climatology")
    b.set_xlabel("Forecast horizon (months ahead)")
    b.set_ylabel("RMSE reduction (%)")
    b.legend(fontsize=8, loc="upper right")
    save(fig, "fig3_rmse_by_horizon.png")

In [ ]:
def permutation_importance(df, split_year=2010, n_repeats=20, seed=0):
    rng = np.random.default_rng(seed)
    model = GradientBoosting().fit(df.loc[: f"{split_year - 1}-12"])
    feats = make_features(df)
    rows = {}
    for h in (1, 3, 6):
        targets = df.loc[f"{split_year}-01":].index
        origins = targets - h
        x = feats.loc[origins]
        y = (df.loc[targets, "level_m"].to_numpy() - df.loc[origins, "level_m"].to_numpy()) * 1000

        def rmse_cm(frame):
            return np.sqrt(np.mean((model.models_[h].predict(frame) - y) ** 2)) / 10

        base = rmse_cm(x)
        for group, cols in FEATURE_GROUPS.items():
            increases = []
            for _ in range(n_repeats):
                shuffled = x.copy()
                shuffled[cols] = x[cols].to_numpy()[rng.permutation(len(x))]
                increases.append(rmse_cm(shuffled) - base)
            rows[(h, group)] = np.mean(increases)
    return pd.Series(rows).unstack(0)

**Drawing Figure 4.** Bar chart of those RMSE increases at 1, 3 and 6 months ahead.

In [ ]:
def fig4_importance(importance):
    order = importance[1].sort_values().index
    fig, axes = plt.subplots(1, 3, figsize=(10, 3.2), sharey=True)
    for a, h in zip(axes, importance.columns):
        a.barh(order, importance.loc[order, h], color=ORANGE, height=0.6)
        a.set_title(f"{h} month{'s' if h > 1 else ''} ahead")
        a.set_xlabel("RMSE increase when shuffled (cm)")
        a.grid(axis="y", visible=False)
        a.axvline(0, color="#c3c2b7", lw=0.8)
    fig.suptitle("What the gradient-boosting model relies on (trained to 2009, scored on 2010-2023)",
                 x=0.01, ha="left", fontsize=10, fontweight="bold")
    fig.tight_layout()
    save(fig, "fig4_feature_importance.png")

**Figure 5.** Case study: the 2013 record low to the 2020 high.

In [ ]:
def fig5_case_study(df, preds):
    fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.8), width_ratios=[1.4, 1])
    window = df.loc["2012-01":"2021-12"]
    a.plot(window.index.to_timestamp(), window["level_m"], color=INK, lw=2.2, label="Observed")
    for name in ["Persistence", "Water balance", "Gradient boosting"]:
        p = preds[(preds["model"] == name) & (preds["horizon"] == 6)
                  & (preds["target"] >= "2012-01") & (preds["target"] <= "2021-12")]
        a.plot(p["target"].dt.to_timestamp(), p["predicted_m"], lw=1.3,
               label=f"{name}, 6 months ahead", **MODEL_STYLE[name])
    a.set_title("2013 low to 2020 high: 6-month-ahead forecasts")
    a.set_ylabel("Level (m, IGLD 1985)")
    a.legend(fontsize=8, loc="upper left")

    # Cumulative supply anomalies relative to the pre-2013 climatology.
    base = df.loc[:"2012-12"]
    clim = base[COMPONENTS].groupby(month_of(base.index)).mean()
    rise = df.loc["2013-02":"2020-07"]
    anom = rise[COMPONENTS] - clim.loc[month_of(rise.index)].to_numpy()
    cum = anom.cumsum() / 10  # cm of water over the lake
    t = rise.index.to_timestamp()
    b.plot(t, cum["precip_mm"], color=BLUE, label="Precipitation")
    b.plot(t, -cum["evap_mm"], color=ORANGE, label="Evaporation (sign flipped)")
    b.plot(t, cum["runoff_mm"], color=AQUA, label="Runoff")
    b.axhline(0, color="#c3c2b7", lw=0.8)
    b.set_title("Cumulative supply anomaly, 2013-2020")
    b.set_ylabel("cm of water over the lake,\nvs. 1950-2012 average")
    b.legend(fontsize=8, loc="upper left")
    save(fig, "fig5_case_study_2013_2020.png")
    return {
        "level_rise_m": df.at[pd.Period("2020-07", "M"), "level_m"]
                        - df.at[pd.Period("2013-01", "M"), "level_m"],
        "cum_precip_cm": cum["precip_mm"].iloc[-1],
        "cum_minus_evap_cm": -cum["evap_mm"].iloc[-1],
        "cum_runoff_cm": cum["runoff_mm"].iloc[-1],
    }